# Análise de Indicadores Econômicos e Mercado Financeiro Brasileiro

**Autor:** Maycon Serzedelo  
**Objetivo:** Explorar a relação entre indicadores macroeconômicos e o desempenho do Ibovespa + modelos simples de previsão.

---

## 1. Configuração e Importações

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import carregar_dados

# Modelos de previsão
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Bibliotecas carregadas com sucesso!')

## 2. Carregamento dos Dados

In [ ]:
df = carregar_dados(data_inicio='2018-01-01')
print('\nShape:', df.shape)
df.tail()

## 3. Evolução Temporal dos Indicadores

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
fig.suptitle('Evolução dos Indicadores Econômicos e Ibovespa', fontsize=16, fontweight='bold')

indicadores = [
    ('IPCA', 'IPCA - Inflação'),
    ('IGP_M', 'IGP-M'),
    ('Selic', 'Selic (%)'),
    ('Cambio_USD_BRL', 'Câmbio USD/BRL'),
    ('IBC_Br', 'IBC-Br (Atividade)'),
    ('PIB', 'PIB'),
    ('Producao_Industrial', 'Produção Industrial'),
    ('Vendas_Varejo', 'Vendas no Varejo'),
    ('Desemprego', 'Taxa de Desemprego (%)'),
    ('Credito_Total', 'Crédito Total'),
    ('Reservas_Internacionais', 'Reservas Internacionais'),
    ('Ibovespa', 'Ibovespa')
]

for ax, (col, titulo) in zip(axes.flat, indicadores):
    if col in df.columns:
        df[col].plot(ax=ax, title=titulo)
        ax.set_xlabel('')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Matriz de Correlação

In [ ]:
plt.figure(figsize=(12, 10))
corr = df.corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True, linewidths=0.5)
plt.title('Matriz de Correlação entre Indicadores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Modelos Simples de Previsão

Vamos testar dois modelos clássicos de séries temporais no **Ibovespa**:

1. **ARIMA** (AutoRegressive Integrated Moving Average)
2. **Prophet** (modelo do Facebook, bom para tendências e sazonalidade)

Usaremos os últimos 12 meses como período de teste.

### 5.1 Preparação dos dados para previsão

In [ ]:
# Série do Ibovespa (apenas dias úteis)
serie = df['Ibovespa'].dropna().copy()

# Divide em treino e teste (últimos 60 dias úteis ≈ 3 meses)
horizonte = 60
treino = serie.iloc[:-horizonte]
teste = serie.iloc[-horizonte:]

print(f'Treino: {treino.index.min().date()} → {treino.index.max().date()} ({len(treino)} pontos)')
print(f'Teste : {teste.index.min().date()} → {teste.index.max().date()} ({len(teste)} pontos)')

### 5.2 Modelo ARIMA

In [ ]:
# ARIMA simples (p=2, d=1, q=2) – valores comuns para séries financeiras
modelo_arima = ARIMA(treino, order=(2, 1, 2))
resultado_arima = modelo_arima.fit()

# Previsão
previsao_arima = resultado_arima.forecast(steps=horizonte)
previsao_arima.index = teste.index

print(resultado_arima.summary().tables[0])

In [ ]:
# Visualização ARIMA
plt.figure(figsize=(12, 5))
plt.plot(treino.index[-120:], treino.values[-120:], label='Treino (últimos 6 meses)', color='steelblue')
plt.plot(teste.index, teste.values, label='Real (Teste)', color='black', linewidth=2)
plt.plot(previsao_arima.index, previsao_arima.values, label='Previsão ARIMA', color='crimson', linestyle='--')
plt.title('Ibovespa – Previsão com ARIMA (2,1,2)', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 Modelo Prophet

In [ ]:
# Prophet exige colunas ds (data) e y (valor)
df_prophet = treino.reset_index()
df_prophet.columns = ['ds', 'y']

modelo_prophet = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True
)
modelo_prophet.fit(df_prophet)

# Cria dataframe futuro
futuro = modelo_prophet.make_future_dataframe(periods=horizonte, freq='B')  # B = business day
forecast = modelo_prophet.predict(futuro)

# Filtra apenas o período de teste
forecast_teste = forecast.set_index('ds').loc[teste.index.min():]

In [ ]:
# Visualização Prophet
plt.figure(figsize=(12, 5))
plt.plot(treino.index[-120:], treino.values[-120:], label='Treino (últimos 6 meses)', color='steelblue')
plt.plot(teste.index, teste.values, label='Real (Teste)', color='black', linewidth=2)
plt.plot(forecast_teste.index, forecast_teste['yhat'], label='Previsão Prophet', color='darkorange', linestyle='--')
plt.fill_between(
    forecast_teste.index,
    forecast_teste['yhat_lower'],
    forecast_teste['yhat_upper'],
    color='orange',
    alpha=0.2,
    label='Intervalo de confiança'
)
plt.title('Ibovespa – Previsão com Prophet', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.4 Comparação de Erros (MAE e RMSE)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ARIMA
mae_arima = mean_absolute_error(teste, previsao_arima)
rmse_arima = np.sqrt(mean_squared_error(teste, previsao_arima))

# Prophet
pred_prophet = forecast_teste['yhat'].reindex(teste.index)
mae_prophet = mean_absolute_error(teste, pred_prophet)
rmse_prophet = np.sqrt(mean_squared_error(teste, pred_prophet))

print('=== Comparação de Performance ===')
print(f'ARIMA   → MAE: {mae_arima:,.0f}  |  RMSE: {rmse_arima:,.0f}')
print(f'Prophet → MAE: {mae_prophet:,.0f}  |  RMSE: {rmse_prophet:,.0f}')

## 6. Insights e Próximos Passos

**O que observar:**
- Nenhum modelo de série temporal pura costuma acertar bem o mercado de ações no curto prazo (mercado é ruidoso).
- O valor deste exercício é aprender o fluxo: preparação → treino → previsão → avaliação.
- Em projetos reais, normalmente combinamos variáveis exógenas (Selic, câmbio, etc.) com modelos mais sofisticados.

**Sugestões para evoluir:**
- Testar diferentes ordens do ARIMA (usar auto_arima)
- Adicionar variáveis exógenas no Prophet (regressors)
- Fazer previsão do IPCA ou da Selic (séries mais “previsíveis” que a bolsa)
- Implementar validação walk-forward